# Gemini 3.1 Pro Preview — Vertex AI Tutorial

A step-by-step guide to using Google's **Gemini 3.1 Pro Preview** model via Vertex AI on Google Colab.

**Course Project ID:** `statg529300220261-01d2`

---

## What you'll learn

1. How to authenticate with Google Cloud from Colab
2. How to call Gemini 3.1 Pro with a **text prompt**
3. How to call Gemini 3.1 Pro with a **multimodal (image) prompt**
4. How to use **Thinking Mode** to control the model's reasoning depth

---

## Model Overview

| Property | Value |
|---|---|
| Model ID | `gemini-3.1-pro-preview` |
| Input types | Audio, images, video, text, PDF |
| Output type | Text |
| Input token limit | 1,000,000 (1M) |
| Output token limit | 64,000 (64K) |
| Release date | 2026-02-19 |
| Stage | Preview |

### Supported Features

- Thinking mode (LOW / MEDIUM / HIGH)
- Grounding with Google Search (Search as a Tool)
- Vertex AI RAG Engine
- URL Context
- Code Execution
- Structured Output
- Context Caching / Implicit Caching
- Batch Prediction
- Provisioned Throughput

---

## Step 1: Authenticate with Google Cloud

Google Colab provides a built-in way to authenticate with your Google Cloud account. Running the cell below will pop up a sign-in window — **sign in with your school email** (the one that has access to the course project).

> **Important:** You must sign in with the account that has access to project `statg529300220261-01d2`.

In [ ]:
from google.colab import auth

# This will open a pop-up asking you to sign in with your Google account.
# Sign in with your school email (e.g., your_uni@columbia.edu).
auth.authenticate_user()

print("Authentication successful!")

### Set your Project ID

Set the project ID for the course. If you have a different project, replace the value below.

In [ ]:
# ============================================================
# CONFIGURATION — Update this if your project ID is different
# ============================================================
PROJECT_ID = "statg529300220261-01d2"
LOCATION = "global"
MODEL_ID = "gemini-3.1-pro-preview"

print(f"Project ID : {PROJECT_ID}")
print(f"Location   : {LOCATION}")
print(f"Model      : {MODEL_ID}")

---

## Step 2: Enable the Vertex AI API

The Vertex AI API must be enabled in your project. You can do this from Colab using `gcloud`.

> If it's already enabled, the command will simply print that it's already active.

In [ ]:
!gcloud config set project {PROJECT_ID}
!gcloud services enable aiplatform.googleapis.com
print("\nVertex AI API is enabled.")

---

## Step 3: Install the Google Gen AI SDK

The `google-genai` SDK provides a unified Python interface for Vertex AI models.

In [ ]:
!pip install --upgrade --quiet google-genai

import google.genai
print(f"google-genai version: {google.genai.__version__}")

### Initialize the Vertex AI Client

This creates a client that all subsequent calls will use. It connects to Vertex AI using your authenticated credentials.

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

print(f"Client initialized for project '{PROJECT_ID}' at location '{LOCATION}'.")

---

## Step 4: Test with curl (Optional)

Before using Python, you can verify the API works by making a raw REST call with `curl`. This is helpful for debugging and understanding the underlying API.

> This step is optional — you can skip ahead to Step 5 if you prefer to use Python directly.

In [ ]:
%%bash -s "$PROJECT_ID" "$MODEL_ID"
PROJECT_ID=$1
MODEL_ID=$2

curl -s \
  -X POST \
  -H "Authorization: Bearer $(gcloud auth print-access-token)" \
  -H "Content-Type: application/json" \
  "https://aiplatform.googleapis.com/v1/projects/${PROJECT_ID}/locations/global/publishers/google/models/${MODEL_ID}:generateContent" \
  -d '{
    "contents": {
      "role": "user",
      "parts": [
        {
          "text": "Explain what generative AI is in 2 sentences."
        }
      ]
    }
  }'

If the above cell returns a JSON response with `"candidates"` containing the model's text answer, the API is working correctly.

---

## Step 5: Text Prompt (Python)

Let's send a simple text prompt to Gemini 3.1 Pro and print the response.

This is the most basic way to interact with the model — you provide a text string and get a text response.

In [ ]:
# ============================================================
# Test 1: Simple text prompt
# ============================================================

response = client.models.generate_content(
    model=MODEL_ID,
    contents="Explain what generative AI is in 3 sentences.",
)

print(response.text)

### Try it yourself

Modify the prompt below and run the cell to ask the model anything:

In [ ]:
# ============================================================
# Your turn! Change the prompt to whatever you like.
# ============================================================

my_prompt = "What are the top 3 applications of GenAI in healthcare?"

response = client.models.generate_content(
    model=MODEL_ID,
    contents=my_prompt,
)

print(response.text)

---

## Step 6: Multimodal Prompt — Image Understanding (Python)

Gemini 3.1 Pro can understand **images, video, audio, and PDFs** alongside text. In this example, we'll send an image from Google Cloud Storage and ask the model to describe it.

We'll use a sample image of blueberry scones provided by Google:
- **GCS URI:** `gs://generativeai-downloads/images/scones.jpg`

In [ ]:
# ============================================================
# Test 2: Multimodal — Image + Text
# ============================================================

# A sample image hosted in Google Cloud Storage
IMAGE_URI = "gs://generativeai-downloads/images/scones.jpg"

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Content(
            role="user",
            parts=[
                types.Part.from_uri(file_uri=IMAGE_URI, mime_type="image/jpeg"),
                types.Part.from_text(text="Describe this image in detail."),
            ],
        )
    ],
)

print(response.text)

### Upload and analyze your own image

You can also upload a local image. The cell below lets you upload a file from your computer and send it to the model.

In [ ]:
import base64
from google.colab import files

# Upload an image from your computer
print("Select an image file to upload:")
uploaded = files.upload()

if uploaded:
    # Get the first uploaded file
    filename = list(uploaded.keys())[0]
    image_bytes = uploaded[filename]

    # Determine MIME type from extension
    ext = filename.rsplit(".", 1)[-1].lower()
    mime_map = {"jpg": "image/jpeg", "jpeg": "image/jpeg", "png": "image/png", "gif": "image/gif", "webp": "image/webp"}
    mime_type = mime_map.get(ext, "image/jpeg")

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=[
            types.Content(
                role="user",
                parts=[
                    types.Part.from_bytes(data=image_bytes, mime_type=mime_type),
                    types.Part.from_text(text="Describe this image in detail."),
                ],
            )
        ],
    )

    print(f"\n--- Analysis of {filename} ---\n")
    print(response.text)
else:
    print("No file uploaded.")

---

## Step 7: Thinking Mode (Python)

Gemini 3.1 Pro is a **thinking model** — it can reason through its thoughts before producing a final answer. You can control the depth of reasoning with the `thinking_level` parameter:

| Thinking Level | Best For | Trade-off |
|---|---|---|
| `LOW` | Simple tasks, chat, fast responses | Minimal reasoning, lowest latency and cost |
| `MEDIUM` | Balanced tasks | Moderate reasoning depth |
| `HIGH` (default) | Complex reasoning, math, coding | Deepest reasoning, higher latency |

> **Note:** Do NOT use the legacy `thinking_budget` parameter together with `thinking_level` — this will return a 400 error.

In [ ]:
# ============================================================
# Test 3: Thinking Mode
# ============================================================

# Try changing this to "LOW", "MEDIUM", or "HIGH"
THINKING_LEVEL = "MEDIUM"

response = client.models.generate_content(
    model=MODEL_ID,
    contents="What is the sum of the first 50 prime numbers?",
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_level=THINKING_LEVEL,
        ),
    ),
)

# Print the thinking process and the final answer
for part in response.candidates[0].content.parts:
    if part.thought:
        print("=== MODEL THINKING ===")
        print(part.text)
        print("=== END THINKING ===\n")
    else:
        print("=== FINAL ANSWER ===")
        print(part.text)

### Compare thinking levels

Let's run the same prompt at all three thinking levels and compare the results and token usage.

In [ ]:
prompt = "A farmer has 17 sheep. All but 9 run away. How many sheep does the farmer have left?"

for level in ["LOW", "MEDIUM", "HIGH"]:
    print(f"\n{'='*60}")
    print(f"  Thinking Level: {level}")
    print(f"{'='*60}")

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(
                thinking_level=level,
            ),
        ),
    )

    # Show token usage
    usage = response.usage_metadata
    print(f"  Prompt tokens   : {usage.prompt_token_count}")
    print(f"  Response tokens : {usage.candidates_token_count}")
    print(f"  Thinking tokens : {usage.thoughts_token_count}")
    print(f"  Total tokens    : {usage.total_token_count}")
    print(f"---")

    # Print final answer only (skip thinking)
    for part in response.candidates[0].content.parts:
        if not part.thought:
            print(f"  Answer: {part.text[:200]}")

---

## Step 8: Examining the Response Object

Let's look at the full response structure to understand what metadata the API returns.

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents="What is 2 + 2?",
)

print("=== Full Response Text ===")
print(response.text)

print("\n=== Usage Metadata ===")
usage = response.usage_metadata
print(f"  Prompt tokens   : {usage.prompt_token_count}")
print(f"  Response tokens : {usage.candidates_token_count}")
print(f"  Thinking tokens : {usage.thoughts_token_count}")
print(f"  Total tokens    : {usage.total_token_count}")

print("\n=== Model Version ===")
print(f"  {response.model_version}")

print("\n=== Finish Reason ===")
print(f"  {response.candidates[0].finish_reason}")

---

## Model Quick Reference

### Media Resolution Defaults

| Resolution Setting | Image tokens | Video tokens | PDF tokens |
|---|---|---|---|
| DEFAULT | 1120 | 70 | 560 |
| LOW | 280 | 70 | 280 |
| MEDIUM | 560 | 70 | 560 |
| HIGH | 1120 | 280 | 1120 |

### Key Changes from Gemini 2.5 Pro

- **Thinking levels** replace `thinking_budget`: use `LOW`, `MEDIUM`, or `HIGH`
- Cannot mix `thinking_budget` and `thinking_level` in the same request (400 error)
- Multimodal function response and streaming function calling support
- Stricter validation on function calling (thought signatures removed in first part)
- PDF token counts now reported under `IMAGE` modality instead of `DOCUMENT`

### Other Gemini Models

| Model | Stage | Best For |
|---|---|---|
| `gemini-3.1-pro-preview` | Preview | Most powerful agentic and coding model |
| `gemini-3-pro-preview` | Preview | Frontier agentic and multimodal |
| `gemini-2.5-pro` | GA | Code and world knowledge |
| `gemini-2.5-flash` | GA | Balanced reasoning and speed |
| `gemini-2.5-flash-lite` | GA | Cost-effective, high throughput |

---

## Troubleshooting

### "Permission denied" or "403 Forbidden"
- Make sure you signed in with the correct account in Step 1
- Re-run the authentication cell at the top
- Verify the Vertex AI API is enabled (Step 2)

### "Model not found" or "404"
- Use the exact model ID: `gemini-3.1-pro-preview`
- Use `location="global"` (not a specific region)

### "400 error" with thinking
- Do NOT mix `thinking_budget` and `thinking_level` in the same request
- Valid thinking levels: `LOW`, `MEDIUM`, `HIGH`

### Import errors
- Re-run the install cell: `!pip install --upgrade google-genai`
- If issues persist, restart the runtime: **Runtime > Restart runtime**, then re-run all cells from the top

### Links

- [Gemini 3.1 Pro in Vertex AI Model Garden](https://console.cloud.google.com/vertex-ai/publishers/google/model-garden/gemini-3.1-pro-preview)
- [Vertex AI Studio](https://console.cloud.google.com/vertex-ai/studio) — Test prompts in the browser
- [Gen AI SDK Documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/sdks/overview)
- [Gemini API Reference](https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/inference)